In [23]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.impute import SimpleImputer
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import BatchNormalization
import shap




In [24]:
df = pd.read_excel("asee_26_data_cleaned.xlsx")

In [25]:
class StudentOutcome_ANN_Classifier:
    def __init__(self, features=None, target="Outcome"):
        self.target = target
        self.features = features or [
            '# of Passed Courses',
            'No. of Failed Courses',
            'Gender',
            'Early College HS?',
            'HS Rank',
            'GPA at Graduation',
            'Credits Transferred',
            'Cal 1 Final Grade (MATH 1411)',
            'Cal 2 Final Grade (MATH 1312)',
            'Statics Final Grade (MECH 1321)',
            'Chemistry Final Grade (CHEM 1305)',
        ]
        self.imputer = None
        self.scaler = None
        self.model = None
        self.history = None
        
    def prepare_data(self, df, test_size=0.2, random_state=42):
        X = df[self.features].copy()
        y = df[self.target].copy()
        
        if y.min() != 0:
            y = y - y.min()
        
        print(f"X shape: {X.shape}")
        print("Outcome distribution:\n", y.value_counts().sort_index())
        
        X_train, X_test, y_train, y_test = train_test_split(
            X, y,
            test_size=test_size,
            random_state=random_state,
            stratify=y
        )
        
     
        self.imputer = SimpleImputer(strategy='median')
        X_train_imputed = self.imputer.fit_transform(X_train)
        X_test_imputed = self.imputer.transform(X_test)
        
        X_train_imputed = pd.DataFrame(X_train_imputed, columns=self.features)
        X_test_imputed = pd.DataFrame(X_test_imputed, columns=self.features)
        
 
        self.scaler = StandardScaler()
        X_train_scaled = self.scaler.fit_transform(X_train_imputed)
        X_test_scaled = self.scaler.transform(X_test_imputed)
        
        return X_train_scaled, X_test_scaled, y_train, y_test
    
    def build_model(self):
        self.model = Sequential([
            Dense(32, activation="relu", input_shape=(len(self.features),)),
            BatchNormalization(),
            Dropout(0.3),
            Dense(16, activation="relu"),
            BatchNormalization(),
            Dropout(0.2),
            Dense(3, activation="softmax")
        ])
        
        self.model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
            loss="sparse_categorical_crossentropy",
            metrics=["accuracy"]
        )
    
    def train(self, X_train_scaled, y_train, class_weight=None, epochs=100, 
              batch_size=32, validation_split=0.2, patience=15):
        if class_weight is None:
            class_weight = {0: 1.0, 1: 2.0, 2: 5.0}
        
        print("Class weights:", class_weight)
        
        early_stop = EarlyStopping(
            monitor="val_loss",
            patience=patience,
            restore_best_weights=True
        )
        
        print("\nTraining neural network...")
        self.history = self.model.fit(
            X_train_scaled, y_train,
            epochs=epochs,
            batch_size=batch_size,
            validation_split=validation_split,
            class_weight=class_weight,
            callbacks=[early_stop],
            verbose=1
        )
    
    def evaluate(self, X_test_scaled, y_test):
        y_pred_prob = self.model.predict(X_test_scaled, verbose=0)
        y_pred = np.argmax(y_pred_prob, axis=1)
        
        print(f"\nTest Accuracy: {accuracy_score(y_test, y_pred):.4f}")
        
        print("\nConfusion Matrix (rows=true, cols=pred):")
        print(confusion_matrix(y_test, y_pred))
        
        print("\nClassification Report:")
        print(classification_report(
            y_test,
            y_pred,
            target_names=[
                "Class 0: Left without degree",
                "Class 1: Degree completion",
                "Class 2: Transferred"
            ],
            digits=3
        ))
        
        return y_pred, y_pred_prob
    
    def fit(self, df, **kwargs):
        """Convenience method to run entire pipeline"""
        X_train, X_test, y_train, y_test = self.prepare_data(df)
        self.build_model()
        self.train(X_train, y_train, **kwargs)
        return self.evaluate(X_test, y_test)


In [26]:
import shap
import matplotlib.pyplot as plt
import numpy as np
import os


class ModelAnalyzer:
    def __init__(self, model, X_train_scaled, X_test_scaled, features, 
                 class_names=None, save_dir=None):
        self.model = model
        self.X_train_scaled = X_train_scaled
        self.X_test_scaled = X_test_scaled
        self.features = features
        self.class_names = class_names or [
            "Class 0: Left without degree",
            "Class 1: Degree completion",
            "Class 2: Transferred"
        ]
        self.save_dir = save_dir or "."
        self.explainer = None
        self.shap_values = None
        
      
        if self.save_dir != ".":
            os.makedirs(self.save_dir, exist_ok=True)
        
    def predict_fn(self, x):
        """Prediction function for SHAP"""
        return self.model.predict(x, verbose=0)
    
    def setup_explainer(self, background_size=50):
        """Initialize SHAP explainer with background data"""
        background = self.X_train_scaled[
            np.random.choice(self.X_train_scaled.shape[0], background_size, replace=False)
        ]
        self.explainer = shap.KernelExplainer(self.predict_fn, background)
        print(f"SHAP explainer initialized with {background_size} background samples")
    
    def compute_shap_values(self, test_sample_size=50):
        """Compute SHAP values for test sample"""
        if self.explainer is None:
            self.setup_explainer()
        
        test_indices = np.random.choice(
            self.X_test_scaled.shape[0],
            min(test_sample_size, self.X_test_scaled.shape[0]),
            replace=False
        )
        X_test_sample = self.X_test_scaled[test_indices]
        
        print(f"Computing SHAP values for {len(test_indices)} test samples...")
        self.shap_values = self.explainer.shap_values(X_test_sample)
        self.X_test_sample = X_test_sample
        
        return self.shap_values
    
    def plot_shap_summary(self, class_idx=None, save=True, dpi=300):
        """Plot SHAP summary for one or all classes"""
        if self.shap_values is None:
            self.compute_shap_values()
        
        if class_idx is not None:
           
            self._plot_class(class_idx, save, dpi)
        else:
          
            for idx in range(len(self.class_names)):
                self._plot_class(idx, save, dpi)
    
    def _plot_class(self, class_idx, save=True, dpi=300):
        """Helper to plot SHAP summary for a single class"""
        class_name = self.class_names[class_idx]
        
        plt.figure(figsize=(12, 8), dpi=150)
        
        shap.summary_plot(
            self.shap_values[class_idx],
            self.X_test_sample,
            feature_names=self.features,
            show=False
        )
        
        plt.title(f'SHAP Feature Impact - {class_name}',
                  fontsize=16, fontweight='bold', pad=20)
        plt.xlabel('SHAP Value (Impact on Model Output)', fontsize=12)
        plt.tight_layout()
        
        if save:
            filename = f'shap_class_{class_idx}_{class_name.lower().replace(" ", "_").replace(":", "")}.png'
            filepath = os.path.join(self.save_dir, filename)
            plt.savefig(filepath, dpi=dpi, bbox_inches='tight', facecolor='white')
            print(f"Saved: {filepath}")
            plt.close()
        else:
            plt.show()
    
    def analyze_all(self, test_sample_size=50, background_size=50, save=True):
        """Run complete analysis pipeline"""
        print("Starting model analysis...")
        self.setup_explainer(background_size)
        self.compute_shap_values(test_sample_size)
        print("\nGenerating SHAP summary plots...")
        self.plot_shap_summary(save=save)
  

In [ ]:
import pandas as pd

classifier = StudentOutcome_ANN_Classifier()
X_train, X_test, y_train, y_test = classifier.prepare_data(df)
classifier.build_model()
classifier.train(X_train, y_train)


analyzer = ModelAnalyzer(
    model=classifier.model,
    X_train_scaled=X_train,
    X_test_scaled=X_test,
    features=classifier.features,
    save_dir="./all_features_shap_results"
)


analyzer.setup_explainer(background_size=50)
analyzer.compute_shap_values(test_sample_size=50)
analyzer.plot_shap_summary()  


X shape: (2460, 11)
Outcome distribution:
 0    1620
1     739
2     101
Name: Outcome, dtype: int64
Class weights: {0: 1.0, 1: 2.0, 2: 5.0}

Training neural network...
Epoch 1/100
50/50 [==============================] - 1s 5ms/step - loss: 1.7954 - accuracy: 0.4663 - val_loss: 0.8974 - val_accuracy: 0.7234
Epoch 2/100
50/50 [==============================] - 0s 2ms/step - loss: 1.1874 - accuracy: 0.6881 - val_loss: 0.5983 - val_accuracy: 0.8655
Epoch 3/100
50/50 [==============================] - 0s 2ms/step - loss: 0.8484 - accuracy: 0.8018 - val_loss: 0.3954 - val_accuracy: 0.9086
Epoch 4/100
50/50 [==============================] - 0s 2ms/step - loss: 0.6898 - accuracy: 0.8577 - val_loss: 0.2749 - val_accuracy: 0.9137
Epoch 5/100
50/50 [==============================] - 0s 2ms/step - loss: 0.6081 - accuracy: 0.8926 - val_loss: 0.2077 - val_accuracy: 0.9442
Epoch 6/100
50/50 [==============================] - 0s 2ms/step - loss: 0.5393 - accuracy: 0.9098 - val_loss: 0.1620 - val_ac

  0%|          | 0/50 [00:00<?, ?it/s]

SHAP explainer initialized with 50 background samples
Computing SHAP values for 50 test samples...


 42%|████▏     | 21/50 [01:12<01:40,  3.48s/it]

In [ ]:
custom_features = [
    'Gender',
    'Early College HS?',
    'HS Rank',
    'GPA at Graduation',
    'Credits Transferred',
    'Cal 1 Final Grade (MATH 1411)',
    'Cal 2 Final Grade (MATH 1312)',
    'Statics Final Grade (MECH 1321)',
    'Chemistry Final Grade (CHEM 1305)',
]
classifier2 = StudentOutcome_ANN_Classifier(features=custom_features)
classifier2.fit(df)

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_graphviz
import graphviz


tree = DecisionTreeClassifier(
    max_depth=4,              
    min_samples_split=20,    
    min_samples_leaf=10,    
    class_weight='balanced', 
    random_state=42
)

tree.fit(X_train, y_train)


y_pred_tree = tree.predict(X_test)
tree_accuracy = accuracy_score(y_test, y_pred_tree)

print(f"Decision Tree Accuracy: {tree_accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred_tree,
    target_names=["Left without degree", "Degree completion", "Transferred"],
    digits=3
))


dot = export_graphviz(
    tree,
    out_file=None,
    feature_names=X.columns,
    class_names=["Left without degree", "Degree completion", "Transferred"],
    filled=True,
    rounded=True,
    special_characters=True
)


graph = graphviz.Source(dot)
graph.render('decision_tree', format='png', cleanup=True)



ValueError: Input contains NaN, infinity or a value too large for dtype('float32').

In [ ]:

def predict_fn(x):
    return model.predict(x, verbose=0)


background = X_train_scaled[np.random.choice(X_train_scaled.shape[0], 50, replace=False)]


explainer = shap.KernelExplainer(predict_fn, background)

test_sample_size = 50
test_indices = np.random.choice(X_test_scaled.shape[0], 
                                min(test_sample_size, X_test_scaled.shape[0]), 
                                replace=False)
X_test_sample = X_test_scaled[test_indices]

shap_values = explainer.shap_values(X_test_sample)

class_names = ["New Class 0: Left without degree ", "New Class 1: Degree completion", "New Class 2: Transferred"]

for class_idx, class_name in enumerate(class_names):
    plt.figure(figsize=(12, 8), dpi=150)
    
    shap.summary_plot(
        shap_values[class_idx],
        X_test_sample,
        feature_names=features,
        show=False
    )
    
    plt.title(f'SHAP Feature Impact - {class_name}', 
              fontsize=16, fontweight='bold', pad=20)
    plt.xlabel('SHAP Value (Impact on Model Output)', fontsize=12)
    plt.tight_layout()
    
    filename = f'shap_class_{class_idx}_{class_name.lower().replace(" ", "_")}.png'
    plt.savefig(filename, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()

100%|██████████| 50/50 [00:43<00:00,  1.15it/s]


In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_graphviz
import graphviz


tree = DecisionTreeClassifier(
    max_depth=4,              
    min_samples_split=20,    
    min_samples_leaf=10,    
    class_weight='balanced', 
    random_state=42
)

tree.fit(X_train, y_train)


y_pred_tree = tree.predict(X_test)
tree_accuracy = accuracy_score(y_test, y_pred_tree)

print(f"Decision Tree Accuracy: {tree_accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred_tree,
    target_names=["Left without degree", "Degree completion", "Transferred"],
    digits=3
))


dot = export_graphviz(
    tree,
    out_file=None,
    feature_names=X.columns,
    class_names=["Left without degree", "Degree completion", "Transferred"],
    filled=True,
    rounded=True,
    special_characters=True
)


graph = graphviz.Source(dot)
graph.render('new decision_tree', format='png', cleanup=True)



Decision Tree Accuracy: 0.8841

Classification Report:
                     precision    recall  f1-score   support

Left without degree      1.000     1.000     1.000       324
  Degree completion      0.950     0.649     0.771       148
        Transferred      0.224     0.750     0.345        20

           accuracy                          0.884       492
          macro avg      0.725     0.800     0.705       492
       weighted avg      0.954     0.884     0.905       492



'new decision_tree.png'

In [ ]:
import dice_ml
from dice_ml import Dice
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from IPython.display import display

target = "Outcome"

features = [
    'Cal 1 Final Grade (MATH 1411)',
    'Cal 2 Final Grade (MATH 1312)',
    'Statics Final Grade (MECH 1321)',
    'Chemistry Final Grade (CHEM 1305)',
]

continuous_features = features  


X = df[features].copy()
y = df[target].copy()

X = X.fillna(X.median())

if y.min() != 0:
    y = y - y.min()

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


preprocessor = ColumnTransformer(
    transformers=[("passthrough", "passthrough", features)],
    remainder="drop"
)

clf = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced"
    ))
])

clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
print(f"Random Forest Accuracy: {accuracy_score(y_test, y_pred):.4f}")

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["Dropped Out", "Completed", "Transferred"],
    digits=3
))


train_df = X_train.copy()
train_df[target] = y_train.values

data_dice = dice_ml.Data(
    dataframe=train_df,
    continuous_features=continuous_features,
    outcome_name=target
)

model_dice = dice_ml.Model(model=clf, backend="sklearn")
dice = Dice(data_dice, model_dice, method="random")


test_df = X_test.copy()
test_df[target] = y_test.values

dropouts = test_df[test_df[target] == 1]
if dropouts.empty:
    raise ValueError("No class-0 examples in test set.")

query_instance = dropouts.iloc[[0]][features]

print("\nQuery Instance (Student Who Dropped Out):")
print(query_instance.T)


permitted_range = {
    'Cal 1 Final Grade (MATH 1411)': [1, 4],
    'Cal 2 Final Grade (MATH 1312)': [1, 4],
    'Statics Final Grade (MECH 1321)': [1, 4],
    'Chemistry Final Grade (CHEM 1305)': [1, 4],
}

cf = dice.generate_counterfactuals(
    query_instance,
    total_CFs=10,
    desired_class=1,
    permitted_range=permitted_range
)


raw_cf_df = cf.cf_examples_list[0].final_cfs_df.copy()

clean_cf_df = raw_cf_df.copy()
for c in features:
    if c in clean_cf_df.columns:
        clean_cf_df[c] = clean_cf_df[c].round().astype(int).clip(1, 4)


pretty_cols = {
    'Cal 1 Final Grade (MATH 1411)': 'Calc I',
    'Cal 2 Final Grade (MATH 1312)': 'Calc II',
    'Statics Final Grade (MECH 1321)': 'Statics',
    'Chemistry Final Grade (CHEM 1305)': 'Chemistry',
    'Outcome': 'Predicted'
}

final_cf_table = clean_cf_df.rename(columns=pretty_cols).reset_index(drop=True)
final_cf_table.insert(0, "CF #", final_cf_table.index + 1)

print("\n" + "="*80)
print("COUNTERFACTUAL SCENARIOS FOR DEGREE COMPLETION")
print("="*80)

display(
    final_cf_table.style
    .set_caption("What grades would these students need to complete their degree?")
    .set_table_styles([
        {"selector": "caption",
         "props": [("caption-side", "top"),
                   ("font-size", "14px"),
                   ("font-weight", "bold")]},
        {"selector": "th",
         "props": [("text-align", "center"),
                   ("background-color", "#f2f2f2")]},
        {"selector": "td",
         "props": [("text-align", "center")]}
    ])
)




  0%|          | 0/1 [00:00<?, ?it/s]

Random Forest Accuracy: 0.6301

Classification Report:
              precision    recall  f1-score   support

 Dropped Out      0.742     0.728     0.735       324
   Completed      0.543     0.466     0.502       148
 Transferred      0.106     0.250     0.149        20

    accuracy                          0.630       492
   macro avg      0.464     0.482     0.462       492
weighted avg      0.656     0.630     0.641       492


Query Instance (Student Who Dropped Out):
                                   2409
Cal 1 Final Grade (MATH 1411)         4
Cal 2 Final Grade (MATH 1312)         0
Statics Final Grade (MECH 1321)       4
Chemistry Final Grade (CHEM 1305)     0


100%|██████████| 1/1 [00:00<00:00,  1.30it/s]


COUNTERFACTUAL SCENARIOS FOR DEGREE COMPLETION


,CF #,Calc I,Calc II,Statics,Chemistry,Predicted
0,1,2,1,4,3,1
1,2,1,1,3,1,1
2,3,4,4,4,1,1
3,4,4,2,4,1,1
4,5,3,1,4,4,1
5,6,3,1,4,3,1
6,7,3,2,4,1,1
7,8,4,1,4,2,1
8,9,2,1,4,2,1
9,10,4,1,4,2,1
